In [1]:
import numpy as np

In [2]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
data = np.array(data)
X = np.array(data[:, :3]).astype(float)
label_map = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}
y = np.array([label_map[row[3]] for row in data])

print(X, y)

[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]] [0 1 2 0 1 2 0 1]


In [3]:
def gini(y):
  classes, count = np.unique(y, return_counts=True)
  total_samples = len(y)
  ps = count / total_samples

  gini = 1 - np.sum(ps ** 2)
  return gini

def entropy(y):
  classes, count = np.unique(y, return_counts=True)
  total_samples = len(y)
  ps = count / total_samples

  entropy = -np.sum(ps * np.log2(ps))
  return entropy

In [4]:
# for feature in features:
#   for threshold in thresholds:
#     try splitting
#     find impurity
#     store it

# use the feature, threshold pair which yields least impurity

In [11]:
def best_split(X, y):
  min_gini = float('inf')
  best_feature = None
  best_threshold = None

  n_features = X.shape[1]

  for feature_index in range(n_features):
    thresholds = np.unique(X[:, feature_index])
    thresholds = (thresholds[:-1] + thresholds[1:]) / 2 # midpoint between consecutive values

    for threshold in thresholds:
      left_mask = X[:, feature_index] <= threshold
      right_mask = ~left_mask
      y_left  = y[left_mask]
      y_right = y[right_mask]

      if len(y_left) == 0 or len(y_right) == 0:
        continue

      n = len(y)
      weighted_gini = (len(y_left) / n)  * gini(y_left) + (len(y_right) / n) * gini(y_right)

      if weighted_gini < min_gini:
        min_gini = weighted_gini
        best_feature_index = feature_index
        best_threshold = threshold

  feature_map = {0: 'Alcohol', 1: 'Sugar', 2: 'Color'}
  print(f"{feature_map[best_feature_index]} <= {best_threshold}")

  return best_feature_index, best_threshold

In [6]:
class Node:
  def __init__(self, feature_index=None, threshold=None, value=None, left=None, right=None):
    self.feature_index = feature_index # index of feature to split on
    self.threshold = threshold # threshold value used for split
    self.left = left # left child
    self.right = right # right child
    self.value = value # predicted class if node is a leaf node

In [15]:
class DecisionTree:
  def __init__(self, max_depth=5, min_samples=2):
    self.max_depth = max_depth
    self.min_samples = min_samples
    self.root = None

  def fit(self, X, y):
    self.root = self.build_tree(X, y, depth=0)

  def build_tree(self, X, y, depth):
    if depth == self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples:
        return Node(value=np.bincount(y).argmax())

    best_feature_index, best_threshold = best_split(X, y)
    left_mask = X[:, best_feature_index] <= best_threshold
    right_mask = ~left_mask

    left_child  = self.build_tree(X[left_mask],  y[left_mask],  depth + 1)
    right_child = self.build_tree(X[right_mask], y[right_mask], depth + 1)

    return Node(feature_index=best_feature_index, threshold=best_threshold, left=left_child, right=right_child)

  def predict(self, X_test):
    return np.array([self.predict_one(x, self.root) for x in X_test])

  def predict_one(self, x, node):
    if node.value is not None:
      return node.value

    if x[node.feature_index] <= node.threshold:
      return self.predict_one(x, node.left)
    else:
      return self.predict_one(x, node.right)

  def print_tree(self, node=None, depth=0):
    if node is None:
      node = self.root

    feature_names = {0: 'Alcohol', 1: 'Sugar', 2: 'Color'}
    label_names   = {0: 'Wine', 1: 'Beer', 2: 'Whiskey'}
    indent = "  " * depth

    if node.value is not None:
      print(f"{indent}-> Predict: {label_names[node.value]}")
      return

    print(f"{indent}if {feature_names[node.feature_index]} <= {node.threshold}:")
    self.print_tree(node.left,  depth + 1)
    print(f"{indent}else:")
    self.print_tree(node.right, depth + 1)

In [8]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])

In [12]:
tree = DecisionTree(max_depth=5, min_samples=2)
tree.fit(X, y)
predictions = tree.predict(test_data)
print(predictions)

Alcohol <= 8.5
Alcohol <= 25.75
[1 2 0]


In [16]:
tree.print_tree()

if Alcohol <= 8.5:
  → Predict: Beer
else:
  if Alcohol <= 25.75:
    → Predict: Wine
  else:
    → Predict: Whiskey
